# [4.1] SC26 Tutorial — Session 2, Part 2: distributing Linalg on the GPU

Part 1 executed Linalg ops on CPU and introduced the tiling transformation. This part uses tiling to distribute the computation across a GPU:

1. **The GPU payloads** — Part 1's `linalg.matvec` and `linalg.matmul`, with their buffers registered for the GPU.
2. **A schedule that distributes** — `tile_using_forall` turns loops into `scf.forall`, whose iterations may run in parallel, and says which GPU ids each one maps onto: an outer one onto GPU **blocks**, a nested one onto the **threads** in each block. The schedule is written for matvec, and the exercise is to modify it for matmul.

The schedule itself builds the `gpu.launch`. Each version is then compiled for this GPU, JIT-executed and checked against numpy.

**This part needs a GPU runtime: Runtime -> Change runtime type -> T4 GPU.**

**Running locally in VS Code instead?** See [`RUNNING_LOCALLY.md`](https://github.com/MaheshRavishankar/sc26_tutorial/blob/main/RUNNING_LOCALLY.md).

## [4.2] Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead can lead to strange errors. To catch up after losing track, `Ctrl+F8` on Colab runs every cell above the current one.

The next cells download the same prebuilt MLIR toolchain as Session 1.

In [ ]:
# [4.3]
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
# [4.4]
import os

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    import requests

    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
# [4.5]
import glob, os, re, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

# The MLIR Python bindings only import into the Python version they were built
# for (3.13, as on Colab). Check that here: with any other version, the next
# cell fails with a confusing "cannot import name 'ir'".
bindings = glob.glob(f"{INSTALL_DIR}/python_packages/mlir_core/mlir/_mlir_libs/_mlir.cpython-*.so")
assert bindings, f"no MLIR Python bindings in {INSTALL_DIR}"
built_for = ".".join(re.search(r"cpython-(\d)(\d+)", bindings[0]).groups())
kernel = f"{sys.version_info.major}.{sys.version_info.minor}"
assert built_for == kernel, (
    f"The MLIR Python bindings are built for Python {built_for}; this kernel is "
    f"Python {kernel}. Switch to a Python {built_for} kernel.")

!mlir-opt --version

In [ ]:
# [4.6]
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

**[4.7]** The next cell clones the tutorial's repo.

In [ ]:
# [4.8]
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # the checkout, for taking a solved file out of one.
    !git clone --depth 1 --no-single-branch https://github.com/{REPO} {REPO_DIR}

**[4.9]** Find the GPU. Its compute capability (`sm_75` on a T4) is the target the kernel gets compiled for.

In [ ]:
# [4.10]
import shutil
import subprocess

assert shutil.which("nvidia-smi"), (
    "no GPU found -- switch the runtime: Runtime -> Change runtime type -> T4 GPU")


def gpu_chip_and_features():
    """Detect the GPU's compute capability and pick a PTX ISA version new
    enough to support it."""
    cc = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc.split(".")
    return f"sm_{major}{minor}", "+ptx78"


GPU_CHIP, GPU_PTX_FEATURES = gpu_chip_and_features()
print("GPU target:", GPU_CHIP, GPU_PTX_FEATURES)

## [4.11] Step 1 — the GPU payloads

Consider the GPU-mapped versions of the two operations:

- **`session2/linalg_matvec_gpu.mlir`** — `@matvec`;
- **`session2/linalg_matmul_gpu.mlir`** — `@matmul`.

They are Part 1's files with one addition. A GPU kernel reads and writes the host's buffers directly, so each buffer is registered with the CUDA driver, with `gpu.host_register`, before the op runs. The `memref.cast` to an unranked memref is there only because `gpu.host_register` takes one.

In [ ]:
# [4.12]
# Opens the files. On Colab, files.view() puts them in the editor pane.
# Running locally there is no google.colab, so this prints the paths instead,
# to open in an editor.
MATVEC_FILE = f"{REPO_DIR}/session2/linalg_matvec_gpu.mlir"
MATMUL_FILE = f"{REPO_DIR}/session2/linalg_matmul_gpu.mlir"

try:
    from google.colab import files

    files.view(MATVEC_FILE)
    files.view(MATMUL_FILE)
except ModuleNotFoundError:
    print(f"open in an editor:\n    {MATVEC_FILE}\n    {MATMUL_FILE}")

**[4.13]** Round-trip both through `mlir-opt`.

In [ ]:
# [4.14]
!mlir-opt {MATVEC_FILE}
assert _exit_code == 0
!mlir-opt {MATMUL_FILE}
assert _exit_code == 0

## [4.15] Step 2 — a schedule that distributes

**`session2/distribute_rows.mlir`** is a transform dialect script, written for matvec. Like Part 1's schedule it matches any Linalg op by interface.

1. The matched op is tiled once, into 4 tiles of 32 rows each, that can be executed in parallel. Each tile is mapped to a block on the GPU.
2. The intra-tile op is tiled again, into 32 tiles of one row each, that can be executed in parallel too, with each row mapped to a thread on the GPU.

`transform.gpu.map_forall_to_blocks` and `transform.gpu.map_nested_forall_to_threads` then convert the nested `scf.forall` loops these tilings produce into a `gpu.launch` with 4 thread blocks of 32 threads each.

The same schedule can be applied to the matmul operation too, but that distributes only the rows of the matrix multiply, with each thread computing a whole row of the output matrix.

The cells below apply it to matvec, then to matmul.

In [ ]:
# [4.16]
# Opens the schedule. On Colab, files.view() puts it in the editor pane.
# Running locally there is no google.colab, so this prints the path instead.
ROWS_SCHEDULE = f"{REPO_DIR}/session2/distribute_rows.mlir"

try:
    from google.colab import files

    files.view(ROWS_SCHEDULE)
except ModuleNotFoundError:
    print(f"open in an editor -- the schedule:\n    {ROWS_SCHEDULE}")

**[4.17]** Round-trip the schedule through `mlir-opt`.

In [ ]:
# [4.18]
!mlir-opt {ROWS_SCHEDULE}
assert _exit_code == 0

### [4.19] Matvec

Apply the schedule to matvec.

In [ ]:
# [4.20]
MATVEC_DISTRIBUTED = "linalg_matvec_gpu_distributed.mlir"

!mlir-opt {MATVEC_FILE} \
    --transform-preload-library=transform-library-paths={ROWS_SCHEDULE} \
    --transform-interpreter \
    -o {MATVEC_DISTRIBUTED}
assert _exit_code == 0

!cat {MATVEC_DISTRIBUTED}

**[4.21]** Compiling and running the transformed code is similar to Session 1 Part 2:

- **`gpu-lower-to-nvvm-pipeline`**, in place of Part 1's conversions to LLVM, compiles the `gpu.launch` body into a GPU binary for this GPU's compute capability and rewrites the host side to launch it;
- **`shared_libs`** pointing at the CUDA runtime wrapper library, and **`engine.initialize()`**, which loads that compiled binary.

In [ ]:
# [4.22]
GPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(convert-linalg-to-loops),"
    "func.func(llvm-request-c-wrappers),"
    "expand-strided-metadata,"
    "lower-affine,"
    f"gpu-lower-to-nvvm-pipeline{{cubin-chip={GPU_CHIP} cubin-format=bin cubin-features={GPU_PTX_FEATURES}}}"
    ")"
)

GPU_SHARED_LIBS = [
    f"{INSTALL_DIR}/lib/libmlir_cuda_runtime.so",
    f"{INSTALL_DIR}/lib/libmlir_runner_utils.so",
    f"{INSTALL_DIR}/lib/libmlir_c_runner_utils.so",
]

REGISTERED = []


def memref_args(*arrays):
    return [ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(x))) for x in arrays]


def run_on_gpu(path, entry_point, args, expected):
    """Lower `path` to LLVM plus a compiled GPU kernel, JIT it, and call
    `entry_point` on fresh copies of `args`. The last one is the output, which
    the op accumulates into, so it starts at zero."""
    ctx = Context()
    module = Module.parseFile(path, ctx)
    PassManager.parse(GPU_LOWERING_PIPELINE, ctx).run(module.operation)
    engine = ExecutionEngine(module, opt_level=3, shared_libs=GPU_SHARED_LIBS)
    engine.initialize()

    args = [x.copy() for x in args]
    REGISTERED.extend(args)
    args[-1][...] = 0
    engine.invoke(entry_point, *memref_args(*args))
    err = float(np.max(np.abs(args[-1] - expected)))
    print(f"{entry_point}: result {args[-1].shape}, max abs error vs numpy = {err:.3g}")
    np.testing.assert_allclose(args[-1], expected, rtol=1e-3, atol=1e-3)
    print("OK")

# The matvec function takes a 128 x 512 matrix and a 512-vector, and writes
# 128 outputs.
ROWS, COLS = 128, 512
rng = np.random.default_rng(0)
mat = rng.random((ROWS, COLS), dtype=np.float32)
vec = rng.random(COLS, dtype=np.float32)
out = np.zeros(ROWS, dtype=np.float32)

run_on_gpu(MATVEC_DISTRIBUTED, "matvec", [mat, vec, out], mat @ vec)

### [4.23] Matmul: Distributing the rows only

Apply the schedule to matmul, unchanged. The `linalg.matmul` inside the `gpu.launch` writes a 1 x 256 slice of `c`: each thread computes a whole row of `c`.

In [ ]:
# [4.24]
MATMUL_ROWS = "linalg_matmul_gpu_rows.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={ROWS_SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_ROWS}
assert _exit_code == 0

!cat {MATMUL_ROWS}

In [ ]:
# [4.25]
M, N, K = 128, 256, 512  # c is M x N; a is M x K; b is K x N
a = rng.random((M, K), dtype=np.float32)
b = rng.random((K, N), dtype=np.float32)
c = np.zeros((M, N), dtype=np.float32)

run_on_gpu(MATMUL_ROWS, "matmul", [a, b, c], a @ b)

### [4.26] Matmul: Distributing the columns too

**Exercise 6: modify `distribute_blocks.mlir`, which starts as a copy of `distribute_rows.mlir`, so that matmul is distributed along columns too — blocks of 32 by 32 threads, with each thread computing one output element `c[i, j]`. Map the columns (`j`) to x and the rows (`i`) to y, for both the blocks and the threads.**

Open the MLIR file by running the cell below. After every edit, round-trip the schedule using the cell after that to check that the schedule is well-formed. Then run the three cells after it, in order.

In [ ]:
# [4.27]
# Opens the file to edit. On Colab, files.view() puts it in the editor pane.
# Running locally there is no google.colab, so this prints the path instead.
BLOCKS_SCHEDULE = f"{REPO_DIR}/session2/distribute_blocks.mlir"

try:
    from google.colab import files

    files.view(BLOCKS_SCHEDULE)
except ModuleNotFoundError:
    print(f"open in an editor -- the file to edit:\n    {BLOCKS_SCHEDULE}")

In [ ]:
# [4.28]
!mlir-opt {BLOCKS_SCHEDULE}
assert _exit_code == 0

**[4.29]** Apply the schedule to matmul.

In [ ]:
# [4.30]
MATMUL_DISTRIBUTED = "linalg_matmul_gpu_distributed.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={BLOCKS_SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_DISTRIBUTED}
assert _exit_code == 0

!cat {MATMUL_DISTRIBUTED}

**[4.31]** **This is the check on the exercise.** The next cell checks the generated code to make sure each thread computes a single element of the result, with the columns on `x` and the rows on `y`.

In [ ]:
# [4.32]
import re

matmul_line = next(line for line in open(MATMUL_DISTRIBUTED) if "linalg.matmul" in line)
shapes = re.findall(r"memref<(\d+x\d+)xf32", matmul_line)
print("shapes the innermost linalg.matmul works on (a, b, c):", shapes)
assert shapes and shapes[-1] == "1x1", (
    "matmul is not distributed along columns yet -- each thread still computes "
    f"a {shapes[-1] if shapes else '?'} slice of c rather than one element")

out_operand = re.search(r"outs\((%\w+)", matmul_line).group(1)
c_slice = next(line for line in open(MATMUL_DISTRIBUTED)
               if line.strip().startswith(f"{out_operand} = memref.subview"))
indices = re.search(r"memref\.subview %\w+\[([^\]]*)\]", c_slice).group(1)
print(f"c's 1x1 slice is cut at [{indices}]")
assert indices.replace(" ", "") == "%thread_id_y,%thread_id_x", (
    "each thread computes one element, but x is not on the columns yet -- map the "
    "column loop to #gpu.block<x> and #gpu.thread<x>, and the row loop to y")

**[4.33]** Run it on the GPU. In the `gpu.launch` above, `blocks(...) in (...)` is 8 by 4: x counts the 8 blocks of columns, y the 4 blocks of rows. Each block is 32 by 32 threads, from `block_dims`. Lowered, the `linalg.matmul` on one row of `a` and one column of `b` leaves only the 512 iterations of `k`.

In [ ]:
# [4.34]
run_on_gpu(MATMUL_DISTRIBUTED, "matmul", [a, b, c], a @ b)

### [4.35] Why x on the columns?

The exercise asked for x on the columns, and the check enforced it. The numpy check would not have: the result is the same with x on the rows. What changes is how the threads touch memory.

`c` is row-major, so `c[i, j]` and `c[i, j+1]` are adjacent. A warp is 32 threads with consecutive `threadIdx.x` and the same `threadIdx.y`. With x on `j`, a warp's 32 threads write `c[i, j]`, `c[i, j+1]`, ... `c[i, j+31]`: 32 adjacent floats, which the GPU **coalesces** into one memory transaction. They also read 32 adjacent elements of a row of `b`, and all read the same `a[i, k]`.

With x on `i` instead, a warp would write `c[i, j]`, `c[i+1, j]`, ... `c[i+31, j]`: 32 floats a row apart, 1 KB at this size, one transaction each. Session 1 Part 2's hand-written kernel made the same choice as this schedule: x on the columns.

The mapping is what decides it, and in the schedule it is one attribute per loop.